# AI Code Reviewer Multi-Agent Test Suite

This notebook tests the multi-agent system powered by **Google Gemini + LangChain + LangGraph**:
1. **Tools**: `read_file`, `search_codebase`, `run_tests`, `git_blame`
2. **Agents**: `SecurityAgent`, `StyleAgent`, `LogicAgent`
3. **Supervisor**: `ReviewSupervisor` (concurrent execution with `asyncio.gather`, deduplication, and deterministic PR verdict)

In [ ]:
# ── 1. Setup & Imports ──────────────────────────────────────────────────
import sys, os, json, asyncio
from pathlib import Path

# Ensure agent-service directory is on sys.path
CWD = Path.cwd()
if str(CWD) not in sys.path:
    sys.path.insert(0, str(CWD))

from dotenv import load_dotenv
load_dotenv(CWD / ".env")

# Enable nested event loop for asyncio in Jupyter
try:
    import nest_asyncio
    nest_asyncio.apply()
except ImportError:
    pass

from app.agents import (
    SecurityAgent,
    StyleAgent,
    LogicAgent,
    ReviewSupervisor,
    Finding,
    ReviewResult,
    compute_verdict,
    deduplicate_findings,
)

print("Environment loaded!")
print("GEMINI_MODEL :", os.getenv("GEMINI_MODEL"))
print("REPO_ROOT    :", os.getenv("REPO_ROOT"))
print("API Key Set  :", bool(os.getenv("GEMINI_API_KEY")))

### Test 1: Deduplication & Verdict Rules (Instant, No API calls)
Verifies deterministic PR verdicts and deduplication across different agent findings.

In [ ]:
# 1. Test Deduplication between Security and Logic on the exact same line
f_security = Finding(
    file="src/auth.py",
    line=25,
    severity="high",
    message="SQL Injection vulnerability: user input directly interpolated in SQL query.",
    category="security"
)
f_logic = Finding(
    file="src/auth.py",
    line=25,
    severity="medium",
    message="User input is directly used in SQL query string concatenation.",
    category="logic"
)
f_style = Finding(
    file="src/auth.py",
    line=10,
    severity="low",
    message="Function name 'DoLogin' should follow snake_case convention.",
    category="style"
)

deduped = deduplicate_findings([f_security, f_logic, f_style])
print(f"Findings before dedup: 3")
print(f"Findings after dedup : {len(deduped)} (Expected: 2)")
assert len(deduped) == 2
assert deduped[0].severity == "high"  # Higher severity preserved
print("Deduplication test: PASSED\n")

# 2. Test Deterministic Verdict Computation
print("Verdict for High finding   :", compute_verdict([f_security]))    # Expected: request_changes
print("Verdict for Medium finding :", compute_verdict([f_logic]))       # Expected: comment
print("Verdict for Low finding    :", compute_verdict([f_style]))       # Expected: approve
print("Verdict for Empty list     :", compute_verdict([]))              # Expected: approve

### Test 2: Tools Verification
Verifies that `read_file` and `search_codebase` operate correctly relative to `REPO_ROOT`.

In [ ]:
from app.tools import read_file, search_codebase

# Test read_file on base_agent.py
read_res = json.loads(read_file.invoke({"path": "app/agents/base_agent.py"}))
print("read_file success :", read_res.get("success"))
print("total_lines       :", read_res.get("total_lines"))

# Test search_codebase for BaseAgent
search_res = json.loads(search_codebase.invoke({"pattern": "class BaseAgent", "max_matches": 5}))
print("search matches    :", search_res.get("total_matches"))
for m in search_res.get("matches", []):
    print(f"  {m['file']}:{m['line']} -> {m['content'].strip()}")

### Test 3: `SecurityAgent` (Real Gemini Call)
Inspects a PR diff containing a hardcoded API secret key and a raw SQL query vulnerability.

In [ ]:
security_pr_diff = """
diff --git a/app/services/auth_service.py b/app/services/auth_service.py
--- a/app/services/auth_service.py
+++ b/app/services/auth_service.py
@@ -15,6 +15,11 @@
+STRIPE_SECRET_KEY = "sk_live_51MzXYZ12345SECRETKEY"
+
 def get_user_profile(user_id):
-    return db.query(User).filter_by(id=user_id).first()
+    # Direct raw query lookup
+    query = f"SELECT * FROM users WHERE id = '{user_id}' AND is_active = 1"
+    return db.execute(query).fetchone()
"""

sec_agent = SecurityAgent()
print("Running SecurityAgent on diff...")
sec_result = await sec_agent.run(security_pr_diff)

print(f"\nSecurity findings ({len(sec_result.findings)}):")
for f in sec_result.findings:
    print(f"- [{f.severity.upper()}] Line {f.line} ({f.category}): {f.message}")

### Test 4: `StyleAgent` (Real Gemini Call)
Reviews code style: wildcard imports, uppercase variable names, and non-Pythonic conventions.

In [ ]:
style_pr_diff = """
diff --git a/app/utils/helpers.py b/app/utils/helpers.py
--- a/app/utils/helpers.py
+++ b/app/utils/helpers.py
@@ -1,5 +1,12 @@
+from os import *
+import sys, json, math, random
+
+def Calculate_Total_Sum(L):
+    X = 0
+    for i in L:
+        X += i
+    return X
"""

style_agent = StyleAgent()
print("Running StyleAgent on diff...")
style_result = await style_agent.run(style_pr_diff)

print(f"\nStyle findings ({len(style_result.findings)}):")
for f in style_result.findings:
    print(f"- [{f.severity.upper()}] Line {f.line} ({f.category}): {f.message}")

### Test 5: `LogicAgent` (Real Gemini Call)
Detects algorithmic bugs: unhandled empty list / zero-division error and an off-by-one boundary index.

In [ ]:
logic_pr_diff = """
diff --git a/app/billing/calculator.py b/app/billing/calculator.py
--- a/app/billing/calculator.py
+++ b/app/billing/calculator.py
@@ -10,6 +10,12 @@
 def calculate_discount_average(discounts):
+    # Bug 1: ZeroDivisionError if discounts is empty
+    total = sum(discounts)
+    average = total / len(discounts)
+    
+    # Bug 2: Off-by-one IndexError on discounts[len(discounts)]
+    for i in range(1, len(discounts) + 1):
+        print(discounts[i])
+    return average
"""

log_agent = LogicAgent()
print("Running LogicAgent on diff...")
logic_result = await log_agent.run(logic_pr_diff)

print(f"\nLogic findings ({len(logic_result.findings)}):")
for f in logic_result.findings:
    print(f"- [{f.severity.upper()}] Line {f.line} ({f.category}): {f.message}")

### Test 6: Full `ReviewSupervisor` End-to-End Concurrent Run
Runs all three agents concurrently with `asyncio.gather()`, aggregates and deduplicates overlapping findings, and computes the deterministic overall PR verdict.

In [ ]:
full_pr_diff = """
diff --git a/app/payment/gateway.py b/app/payment/gateway.py
new file mode 100644
--- /dev/null
+++ b/app/payment/gateway.py
@@ -0,0 +1,24 @@
+import os
+
+API_PRIVATE_KEY = "AIzaSyD-FakeSecretKey_987654321"
+
+def Process_Payment(card_number, amount, user_id):
+    if amount == 0:
+        # Logic error: negative amounts are not blocked
+        return False
+
+    # Security & Logic: SQL injection vulnerability
+    query = "UPDATE accounts SET balance = balance - " + str(amount) + " WHERE user_id = '" + user_id + "'"
+    db_conn.execute(query)
+    
+    return True
"""

supervisor = ReviewSupervisor()
print("Supervisor executing SecurityAgent, StyleAgent, and LogicAgent concurrently via asyncio.gather()...\n")

final_result = await supervisor.review(full_pr_diff)

print("=" * 60)
print(f"OVERALL PR VERDICT: {final_result.verdict.upper()}")
print("=" * 60)

if final_result.errors:
    print("\nErrors encountered:", final_result.errors)

print(f"\nTotal Aggregated & Deduplicated Findings: {len(final_result.findings)}")
for i, finding in enumerate(final_result.findings, 1):
    print(f"\n{i}. [{finding.severity.upper()}] {finding.file}:{finding.line} ({finding.category})")
    print(f"   {finding.message}")